# 89. 스마트 그리드 정전 지속 시간 분석 실습

> **📥 [실습 주피터 노트북(.ipynb) 다운로드](practice.ipynb)**
## 실전 데이터 분석 89: 전력망 관리 섹터별 예반 보수 점검 주기 대비 정전 지속 시간 및 피해 규모 분석

![도입 만화](img/intro_comic.png)

### 📊 실습 개요 (Tutorial Overview)
도시 스마트 그리드(Smart Electric Grid) 전력망의 고장정비 송전 정전 이력 데이터셋입니다. 각 관리 섹터(GridSector)의 기상 악화(WeatherAnomalyIndex), 설비 점검 예방 주기(MaintenanceCycle_Months)가 실제 정전 복구 시간(OutageDuration_Hours) 및 피해 가구 수(CustomersAffected)에 미치는 위험 요인을 진단하고 시각화합니다.

---

### 🛠️ 핵심 분석 실습 역량 (Core Skills)
* **이상치 억제 중앙값 대치 (\`fillna\`):** 기록 누락된 점검 정비 주기를 극단값 왜곡이 적은 중앙값(Median)으로 대치 처리합니다.
* **부서별 정전 지연 및 가구 피해 상관 분석 (\`barplot\`, \`scatterplot\`):** 관리 섹터별 일평균 정전 복구 시간 대조 및 정전 시간 대비 피해 가구 규모의 기상 연계 시각화를 완성합니다.

---

## 🧐 실무 도메인 지식 가이드 (Domain Knowledge Guide)

> **🔋 친환경 에너지 및 환경 (Energy & Environment)**
> 재생 에너지 발전 성능 예측, 스마트 그리드 수요 관리, 탄소 배출량 모니터링 등 지속 가능한 환경 인프라를 위한 데이터 분석 분야입니다.
>
> * **수요와 공급 매칭(Demand/Supply)**: ESS(에너지저장장치) 배터리 충전 셀 온도 편차와 전력 소비 피크 구간을 연계해 그리드 안전성을 모니터링합니다.
> * **재생 에너지의 외생 변수 의존성**: 풍속, 일사량, 패널 먼지 오염도 등 환경 요인이 발전 효율 하락에 기여하는 회귀식을 추계합니다.
> * **기후 위험 지수 진단**: 해수면 온도 변화나 산불 위험 요인 등 환경적 이상치를 공간 융합 통계 기법으로 분석합니다.

---

## Step 1: 데이터 불러오기 및 기본 정보 확인 (Data Load)

![Step 1 데이터 수집 개념도](img/step1_dataset.svg)

제공된 CSV 파일을 분석 컴퓨터로 불러와 로드하고, 컬럼의 타입 및 데이터 구조를 확인합니다.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 한글 폰트 설정
import koreanize_matplotlib
sns.set_theme(style="whitegrid")

# 데이터 로드
df = pd.read_csv('./electric_grid.csv')
print(df.info())
print(df.head())


RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   OutageID                 1000 non-null   int64  
 1   GridSector               1000 non-null   str    
 2   WeatherAnomalyIndex      1000 non-null   float64
 3   MaintenanceCycle_Months  986 non-null    float64
 4   OutageDuration_Hours     1000 non-null   float64
 5   CustomersAffected        1000 non-null   int64  
dtypes: float64(3), int64(2), str(1)
memory usage: 47.0 KB
> 
> OutageID GridSector  WeatherAnomalyIndex  MaintenanceCycle_Months  OutageDuration_Hours  CustomersAffected
0    890001   Sector C                 0.64                     15.0                   2.8               1288
1    890002   Sector C                 5.55                      6.0                  11.1               5355
2    890003   Sector D                 0.02                     10.0                   2.6               3247
3    890004   Sector B                 6.89                      5.0                  12.8              15000
4    890005   Sector A                 8.03                     12.0                  15.9               2421
> ```

---

## Step 2: 결측치 및 데이터 정제 (Data Cleaning)

![Step 2 데이터 정제 개념도](img/step2_missing_values.svg)

수집 과정에서 빈칸으로 기록된 결측값(NaN)의 존재 여부를 진단하고, 데이터 도메인 성격에 부합하는 통계 수치 대입 기법으로 정제합니다.


In [ ]:
# 1. 컬럼별 결측치 개수 확인
print("--- 정제 전 결측치 확인 ---")
print(df.isnull().sum())

# 2. 결측치 전처리 및 대치 수행
median_maint = df['MaintenanceCycle_Months'].median()
df['MaintenanceCycle_Months'] = df['MaintenanceCycle_Months'].fillna(median_maint)
print(df.isnull().sum())


GridSector                  0
WeatherAnomalyIndex         0
MaintenanceCycle_Months    14
OutageDuration_Hours        0
CustomersAffected           0
> 
> --- 정제 후 결측치 확인 ---
> OutageID                   0
GridSector                 0
WeatherAnomalyIndex        0
MaintenanceCycle_Months    0
OutageDuration_Hours       0
CustomersAffected          0
> ```

### 💡 분석가의 통찰 (Analyst's Insight)
* **점검 주기 중앙값 대치의 인프라 비즈니스 근거:** 설비 보수 주기는 대부분 3개월 내지 6개월 단위의 일정 주기에 쏠리지만, 극히 드물게 관리 소홀 섹터의 수십 개월 정비 지연이 잡히는 비대칭 구조를 갖습니다. 평균치를 적용해 대입하면 정상 관리 부문의 결측치가 실제보다 방만하게 과대 평가되므로 중간값 대치가 올바른 정형화 방식입니다.

---

## Step 3: 단변수 분포 분석 (Univariate EDA)

![Step 3 시각화 개념도](img/step3_visualization.svg)

가장 먼저 핵심 변수가 전체 데이터에서 어떤 빈도와 분포를 가졌는지 단일 변수 시각화를 통해 파악해 봅니다.


In [ ]:
plt.figure(figsize=(8, 5))

# 단변수 분석 그래프 생성
sns.barplot(data=df, x='GridSector', y='OutageDuration_Hours', palette='Set1', errorbar=None)
plt.title('전력망 관리 섹터별 평균 정전 복구 소요 시간 (시간)', fontsize=14, fontweight='bold')
plt.show()


### 💡 시각화 차트 읽는 법 & 인사이트
* **노후 관리 섹터의 기저 복구 지연 규명:** 각 행정 파트의 복구 시간 막대를 비교하면 특정 섹터(예: Sector_C)의 평균 정전 지연 시간이 현저하게 치솟아 있습니다. 해당 구역의 송배전 배선 및 비상 대기 인프라 재점검과 예산 우선 투입이 요구됨을 강력히 대변합니다.

---

## Step 4: 다변수 상관관계 및 이상치 분석 (Multivariate EDA)

![Step 4 상관관계 분석 개념도](img/step4_multivariate.svg)

두 개 이상의 변수를 동시에 결합하여, 조건에 따른 수치 차이나 독립 변수와 종속 변수 간의 통계적 경향을 분석합니다.


In [ ]:
plt.figure(figsize=(9, 6))

# 다변수 분석 그래프 생성
sns.scatterplot(data=df, x='OutageDuration_Hours', y='CustomersAffected', hue='WeatherAnomalyIndex', palette='coolwarm', alpha=0.8)
plt.title('정전 복구 지속 시간 대비 누적 피해 가구 수와 기상 이변 상관성', fontsize=14, fontweight='bold')
plt.show()


### 💡 코드 딥다이브 & 비즈니스 통찰 (Analyst's Insight)
* **정전 장기화에 따른 피해 규모의 지수 폭발 및 기상 재난 동조:** 정전 지속 시간(X축)과 피해 가구(Y축)는 단순 선형이 아니라 정전 시간이 늘어날수록 피해자 규모가 위로 둥글게 치솟는 양상을 띱니다. 특히 붉은색 계열(기상이변 지수가 매우 높은 날) 정전 발생 시, 복구 인력 접근 불가로 인해 정전이 장기화되며 광범위한 지역 블랙아웃(피해 가구 수직 상승)으로 동조화되는 재난 인과 흐름을 명백히 보여줍니다.

---

## Step 5: 통계적 직관과 해석 (Statistical Logic)

> 💡 **[망 신뢰성과 고장 간격(MTBF) 및 복구 시간(MTTR)의 데이터 지표화]**
... (생략: 전력망 시스템 신뢰도 및 MTBF/MTTR 개념 요약)

---

## 🎯 30분 강의 마무리 및 심화 과제

오늘 우리는 실전 데이터셋을 분석하여 판다스로 데이터를 가공 및 정제하고, 시각화를 활용하여 핵심 변수 간의 통계적 유의성을 검증했습니다. 데이터 속에서 숨겨진 패턴을 올바른 시각으로 탐색하는 능력이 데이터 사이언티스트의 가장 강력한 무기입니다.

### 📝 심화 과제 (Advanced Challenge)
* **예방 보수 주기(MaintenanceCycle_Months)와 정전 복구 시간의 상관 분석:** 설비 관리를 오래 방치할수록 고장 시 복구 지연이 악화되는지 두 요인의 상관계수를 산출해 보세요.
* **도시 광역 블랙아웃 위험군 요약:** 정전 시간이 8시간 이상 지속되고 피해 가구가 5,000세대를 넘는 대형 정전 이력에 속한 섹터별 특징을 요약 출력해 보세요.
